In [0]:
import time

# Landing path and configuration
landing_path = "/Volumes/workspace/shprod/payrolldata/data/"
expected_file_name = "payrolldata.csv"

# Polling parameters (wait up to 60 minutes, checking every 60 seconds)
timeout_seconds = 3600
poll_interval = 60
elapsed_time = 0
file_found = False
target_file_path = ""

print(f"Checking for files in: {landing_path}")

while elapsed_time < timeout_seconds:
    try:
        files = dbutils.fs.ls(landing_path)
        # Find matching non-empty files
        valid_files = [
            f.path for f in files 
            if f.path.endswith(expected_file_name) and f.size > 0
        ]
        
        if valid_files:
            target_file_path = valid_files[0]
            file_found = True
            print(f"File detected: {target_file_path}")
            break
            
    except Exception as e:
        print(f"Waiting for path initialization... ({e})")

    time.sleep(poll_interval)
    elapsed_time += poll_interval
    print(f"Waiting for file... ({elapsed_time}s / {timeout_seconds}s)")

if not file_found:
    # Fail task or skip downstream execution gracefully
    raise FileNotFoundError(f"No valid file arrived in {landing_path} within {timeout_seconds} seconds.")

# Pass the detected file path to downstream tasks (01_ingest_and_validate)
dbutils.jobs.taskValues.set(key="landed_file_path", value=target_file_path)

Checking for files in: /Volumes/workspace/shprod/payrolldata/data/
File detected: dbfs:/Volumes/workspace/shprod/payrolldata/data/payrolldata.csv
